# 2 · Creating Geometry and Meshes

We **make geometry ourselves** and mesh it: a **3D** Toblerone, **2D** sketches (a sword and a
shield) with the meshing knobs, **named regions** in 3D, and **1D** meshes (plus a few
supplements).

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from netgen.meshing import MeshingParameters
from ngsolve import * #Mesh, H1, GridFunction, x, y, VOL, BND, Integrate, CF
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from math import sqrt

## 1. Feed the Beast

Unit 1 ended with a promise: to win the Beast over, you first bring it **food**. 
So before we arm ourselves we **build a chocolate bar** from scratch, and meet the
core **OpenCASCADE (OCC)** workflow NGSolve ships through Netgen.

In [ ]:
# the bar, parametrised — change any number and re-run
def trapezoid(a, b, c, d): # a planar quad face from 4 points
    return Face(Wire([Segment(a, b), Segment(b, c), Segment(c, d), Segment(d, a)]))

view = dict(euler_angles=[-70, 0, -20])                          # a nice 3/4 view

def peak(cx, cy, z0, bx, by, h, s=0.15, draw=False):
    """two intersecting roofs: centered around (cx, cy, z0), """
    tr_xz = trapezoid(Pnt(cx-  bx/2, cy-by/2,   z0), Pnt(  cx+bx/2, cy-by/2,   z0),
                      Pnt(cx+s*bx, cy-by/2, z0+h), Pnt(cx-s*bx, cy-by/2, z0+h) )
    tr_xz.col = (1,0,0) # red
    roof_xz_y = Prism(tr_xz,Vec(0, by, 0)); roof_xz_y.faces.col = (1,0.5,0.5,0.75)

    tr_yz = trapezoid(Pnt(cx-bx/2,   cy-by/2,   z0), Pnt(cx-bx/2,   cy+by/2,   z0),
                      Pnt(cx-bx/2, cy+s*by, z0+h), Pnt(cx-bx/2, cy-s*by, z0+h) )
    tr_yz.col = (0,0,1) # blue
    roof_yz_x = Prism(tr_yz,Vec(bx, 0, 0)); roof_yz_x.faces.col = (0.5,0.5,1,0.75)

    pk = roof_xz_y * roof_yz_x
    pk.faces.col = (0.3,0,0.7,1)
    if draw:
        labels = [
            {"type": "text", "text": "trapezes", "position": [0, -1, -1]},
            {"type": "text", "text": "overlapping prisms", "position": [3, -1, -1]},
            {"type": "text", "text": "intersection", "position": [6, -1, -1]},
        ]
        Draw( Glue([Glue([tr_xz, tr_yz]),
                    Glue([tr_xz, roof_xz_y, tr_yz, roof_yz_x, pk]).Move((3,0,0)), 
                    Glue([tr_xz, tr_yz, pk]).Move((6,0,0)), 
                   ]), objects=labels, **view )
    return pk

peak(0, 0, 0, 2, 3, 2.6, s=0.15, draw=True)

**Assemble, then round and cut.** 
* a flat **base** (`Box`)
* we add `n_peaks` peaks with `+` (union).
* then **cut** the front and back flat with an intersection (`*`)
* round the **valley** edges with `MakeFillet`,
* and paint it chocolate brown.

In [ ]:
b, depth, H = 2.0, 3.0, 2.2
n_peaks = 3
peaki = lambda i: peak((0.85*i + 0.5) * b, depth/2, 0, b, depth, H, s=0.15)
solid = sum([peaki(i) for i in range(1,n_peaks)],start=peaki(0))
W = (0.85*n_peaks+0.15) * b # total length
Draw(solid, **view)

solid *= Box(Pnt(0+0.4*b, 0, 0.1*H), Pnt(W-0.4*b, depth, H))
valleys = [e for e in solid.edges if e.center[2] < 0.5*H and e.center[2] > 0.2*H ]
solid = solid.MakeFillet(valleys, 0.15)                         # round only the valley profile
solid.faces.col = (0.42, 0.26, 0.16) # chocolate brown
Draw(solid, **view)

**Ready to mesh.** One line turns the OCC solid into an `OCCGeometry` — the input to the Netgen
mesher; the generous fillets let us use a fairly **coarse** `maxh`. Rotate the result: recognise
it? It is, of course, a **Toblerone** 🍫. Now to the **forge**.

In [ ]:
mesh_tob = Mesh(OCCGeometry(solid).GenerateMesh(maxh=2.0))
print(f"the mesh: {mesh_tob.ne} tetrahedra, {mesh_tob.nv} vertices")
Draw(mesh_tob, **view)

## 2. Forge a sword — a 2D sketch

A `WorkPlane` is a pen on a sheet of paper:
* `MoveTo` puts it down,
* each `LineTo` draws a straight segment,
* `Close` joins back to the start,
* and `.Face()` fills the closed outline.

We trace the **silhouette of a sword** in one loop — blade, crossguard, grip, pommel. Here is the exact path first, so you can read the
code below as a walk from point to point:

![The sword outline as 14 numbered points traced counter-clockwise into one closed loop](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/sword-sketch.png)

> **Important**
>
> **Orientation matters.** A face's outline must run **counter-clockwise** (so it encloses a
> *positive* area). Trace it clockwise and the area comes out negative — the mesher then
> silently fails to triangulate it. We walk **tip → left → pommel → right → back** (points
> 0 → 13 in the sketch).

In [ ]:
def sword_2d():
    """Silhouette of a sword as one counter-clockwise outline (the 14 points of the sketch)."""
    return (WorkPlane()
            .MoveTo(0, 10)                                            # 0: the tip
            .LineTo(-0.5, 2.2).LineTo(-2.2, 2.2).LineTo(-2.2, 1.5)    # 1–3: left blade edge → guard
            .LineTo(-0.35, 1.5).LineTo(-0.35, -1.6)                   # 4–5: into the grip, down
            .LineTo(-0.95, -2.2).LineTo(0, -2.95).LineTo(0.95, -2.2)  # 6–8: the diamond pommel
            .LineTo(0.35, -1.6).LineTo(0.35, 1.5)                     # 9–10: back up the grip
            .LineTo(2.2, 1.5).LineTo(2.2, 2.2).LineTo(0.5, 2.2)       # 11–13: guard → right blade edge
            .Close().Face())

sword = sword_2d()
print(f"the sword: area {sword.mass:.1f}  (positive → correctly oriented)")
mesh_sw = Mesh(OCCGeometry(sword, dim=2).GenerateMesh(maxh=0.4))
print(f"meshed: {mesh_sw.ne} triangles")
Draw(mesh_sw)

## 3. A shield — curved edges and `mesh.Curve`

`Spline` draws a smooth curve through points — here the rounded sides of a **shield**. We trace it the same **sketch → face** way as the sword:

![The heater-shield outline — flat top, straight sides, two splines curving to the bottom point, as 7 numbered points](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/shield-sketch.png)

The one new thing is **`mesh.Curve(order)`**: straight-sided elements approximate a curve by a
**polygon**, and `Curve` *bends* the element edges to follow the true boundary.

In [ ]:
def shield_2d(W=3.0, H=3.5):
    """A heater shield: flat top, sides splining down to a point."""
    return (WorkPlane()
            .MoveTo(-W, H).LineTo(-W, 0.3)                           # top-left, down left side
            .Spline([(-W*0.6, -2.0), (0, -H)])                       # curve to the bottom point
            .Spline([(W*0.6, -2.0), (W, 0.3)])                       # curve up the right side
            .LineTo(W, H).Close().Face())                            # up right, close the top

shield = shield_2d()
mesh_sh = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=1))
mesh_sh.Curve(3)                                                     # bend elements onto the curve
Draw(mesh_sh)

### More meshing options — `MeshingParameters`

`maxh` and `Curve` are the two you reach for most, but Netgen exposes many more through a
**`MeshingParameters`** object (or directly as keyword arguments to `GenerateMesh`). A few
worth knowing:

* **`minh`** 
* **`grading`** (0–1) — how fast the element size may change from one element to the next.
  Small (≈0.1) ⇒ smooth, gradual size changes (more elements); large (≈0.9) ⇒ abrupt.
* **`segmentsperedge`**, **`optsteps2d`**, **`optsteps3d`**, **`delaunay`**, **`quad_dominated`**, ..

execute `MeshingParameters?` for more (although `minh` seems to be a hidden feature)

In [ ]:
#MeshingParameters?

In [ ]:
for g in (0.7, 0.2):
    sw = sword_2d()
    sw.edges.Nearest((0.2,3.2)).maxh = 0.1                                       # a fine feature at the tip
    m = Mesh(OCCGeometry(sw, dim=2).GenerateMesh(maxh=3, grading=g))
    note = "gradual → fine zone spreads" if g < 0.5 else "abrupt → fine zone stays local"
    print(f"grading={g}:  {m.ne:4d} triangles  ({note})")
    Draw(m)

**Local refinement.** You rarely want the *whole* mesh fine. Setting **`.maxh` on a single
sub-shape** refines only there — here the **sharp tip** of the blade, where the geometry is
thin and the solution will vary fastest, while the broad grip stays coarse.

In [ ]:
sword_ref = sword_2d()
sword_ref.vertices.Max(Y).maxh = 0.0002 # fine only near the tip 
mesh_ref = Mesh(OCCGeometry(sword_ref, dim=2).GenerateMesh(minh=0.6))
print(f"locally refined sword: {mesh_ref.ne} triangles (fine tip, coarse grip)")
Draw(mesh_ref)

## 4. Named regions — materials & boundaries, in 3D

For real models **named parts** are useful. 

Let's go 3D again with `face.Extrude(d)`. 

Crucially, we **name** them: 
* a solid's `.name` is its **material** (a *volume* region),
* and `.faces.name` names its surface as a **boundary** region.

These names are exactly how every later unit attaches **material coefficients** and **boundary conditions**.

In [ ]:
sword3d  = sword_2d().Extrude(0.5)
shield3d = shield_2d().Extrude(1.0).Move((0, 0, -3))                 # shield set behind the blade
sword3d  = sword3d.Rotate(Axis(Pnt(0, 0, 0), Z), 25).Move((1.5, -1, -6))

sword3d.name  = "sword"                                             # material name (a volume region)
shield3d.name = "shield"
sword3d.faces.name  = "sword_surf"                                 # boundary name (all its faces)
shield3d.faces.name = "shield_surf"

scene = Glue([sword3d, shield3d])
mesh3d = Mesh(OCCGeometry(scene).GenerateMesh(maxh=1.2)); mesh3d.Curve(2)
print(f"the 3D loadout: {mesh3d.ne} tetrahedra")
Draw(mesh3d)

**Inspecting regions.** Ask the mesh what it carries: `GetMaterials()` / `GetBoundaries()`
list the names, while `mesh.Materials(pattern)` / `mesh.Boundaries(pattern)` select a
**`Region`** (a mask over elements) by a **regex** pattern — `"sword"`, `"sword|shield"`,
`".*"`. Each element also knows its own material as `el.mat`.

In [ ]:
print("materials :", mesh3d.GetMaterials())
print("boundaries:", mesh3d.GetBoundaries())

In [ ]:
# each element knows its own material, so we can count per sub-domain:
n_sword  = sum(1 for el in mesh3d.Elements(VOL) if el.mat == "sword")
n_shield = sum(1 for el in mesh3d.Elements(VOL) if el.mat == "shield")
print(f"tetrahedra — sword: {n_sword}, shield: {n_shield}, total: {mesh3d.ne}")

n_sword_surf  = sum(1 for el in mesh3d.Elements(BND) if el.mat == "sword_surf")
n_shield_surf = sum(1 for el in mesh3d.Elements(BND) if el.mat == "shield_surf")
print(f"triangles — sword_surf: {n_sword_surf}, shield_surf: {n_shield_surf}, total: {mesh3d.GetNE(BND)}")

In [ ]:
help(mesh3d.Materials("sword"))

print([(key, str(mesh3d.Materials(key).Mask())) for key in ["sword", ".*", "shield", ".*o.*", "bow"]])
print([(key, str(mesh3d.Boundaries(key).Mask())) for key in ["sword_surf", ".*", "shield_surf", "bow_surf"]])

vol_sword = Integrate(1, mesh3d, definedon=mesh3d.Materials("sword"))
print(f"volume of the 'sword' region: {vol_sword:.6f} ({sword3d.mass:.6f})")

In [ ]:
for i, el in enumerate(mesh3d.Materials("shield").Elements()):
    print(f"{i:4}:: element number {el.nr}, index: {el.index}, mat: {el.mat}, vertices: {el.vertices}, faces: {el.faces}, facets: {el.facets}, edges: {el.edges}")
    if i > 9:
        print("       ...")
        break

---
## Supplementary A — 1D meshes

*(Supplementary.)* A mesh can be **one-dimensional** too — a chain of intervals on $[0,1]$.
`Make1DMesh(n)` spaces $n$ of them **uniformly**; a `mapping` grades them, packing points
where you need resolution (say near a boundary layer).

In [ ]:
def nodes(m): return sorted(p[0] for p in m.ngmesh.Points())
uniform = Make1DMesh(12)
graded  = Make1DMesh(12, mapping=lambda t: t**1.7)                  # clustered near x=0
fig, ax = plt.subplots(figsize=(7, 1.4))
ax.plot(nodes(uniform), [1]*13, "o-", label="uniform")
ax.plot(nodes(graded),  [0]*13, "o-", label="graded $t^{1.7}$")
ax.set_yticks([0, 1]); ax.set_yticklabels(["graded", "uniform"]); ax.set_xlabel("x")
ax.legend(loc="center right"); ax.set_ylim(-0.5, 1.5); fig.tight_layout()

---
## Supplementary B — an adaptive refinement step

*(Supplementary.)* Instead of refining everywhere, we can **mark** individual elements and
refine only those. A real adaptive loop marks by an **error estimator** (unit 10); here we
use a toy criterion — **proximity to a chosen world point** — to show the mechanics. We set
NGSolve's per-element **refinement flag** with `mesh.SetRefinementFlag(el, …)`, then call
**`mesh.Refine()`**, which bisects the marked elements (plus a small conforming closure).
See the i-tutorial
[1.6 Adaptivity](https://docu.ngsolve.org/latest/i-tutorials/unit-1.6-adaptivity/adaptivity.html).

In [ ]:
mesh_ad = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.6))
focus = (0.0, -1.0)                                                 # refine near the shield's tip
for step in range(3):                                              # three adaptive sweeps
    for el in mesh_ad.Elements(VOL):
        cx, cy = (sum(c) / 3 for c in zip(*[mesh_ad[v].point for v in el.vertices]))
        mesh_ad.SetRefinementFlag(el, sqrt((cx - focus[0])**2 + (cy - focus[1])**2) < 0.2)
    mesh_ad.Refine()
print(f"after 3 marked refinements near {focus}: {mesh_ad.ne} triangles (fine only there)")
Draw(mesh_ad)

---
## Supplementary C — importing a real external model

*(Supplementary.)* Geometry need not be sketched by hand — NGSolve reads common CAD/mesh
formats. Here we **import an `.stl` model** (a surface triangulation — a blocky *Minecraft
sword*) and let Netgen build a **volume mesh** from it. The same `OCCGeometry` route reads
**STEP/IGES/BREP** CAD files, and imported parts can be **combined** with sketched ones via
the boolean operators (`+`, `-`, `*`, `Glue`) you have already met.

In [ ]:
from netgen.stl import STLGeometry
# Colab opens only the .ipynb, so the sibling data/ folder isn't there. Fetch the model
# from the colab mirror (CI ships notebooks/data to that branch, like the images). A
# no-op wherever the file already exists: local Jupyter, JupyterLite, the static build.
import os, urllib.request
_stl = "data/minecraft-sword.stl"
if not os.path.exists(_stl):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/" + _stl, _stl)
imported = STLGeometry(_stl)
mesh_mc = Mesh(imported.GenerateMesh(maxh=15))                      # coarse: the model is detailed
print(f"imported Minecraft sword: {mesh_mc.ne} tetrahedra, {mesh_mc.nv} vertices")
Draw(mesh_mc)

**Armed and fed.** You built a **3D** Toblerone, sketched a sword & shield in **2D** and tuned
the meshing knobs, named and inspected regions in 3D, and dropped to **1D** — plus the
supplements (query a mesh, refine it adaptively, import a model). Next we meet the one object
NGSolve evaluates *everywhere* — the **CoefficientFunction**.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("01-meet-the-beast", "1 · Meet the Beast")
    _next = ("03-coefficientfunctions", "3 · What is a CoefficientFunction? 🔨")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))